# The eyes, served from a free T4

Our grounder (Qwen2-VL-2B + the step-1179 adapter) in **the setup the eval scored** (fp16 on a T4,
1568 px), behind a token-guarded HTTP server and a public Cloudflare tunnel. The laptop's agent loop
sends a screenshot + a description, and gets a point back (`src/computeruse/eyes_remote.py`).

**Run it:** Session options → Accelerator **GPU T4 x2**, Internet **on**. Run all, **interactively**
(keep the tab open: the session lives as long as the tab, up to 12 h). The last cell prints the two
`setx` lines to run on the laptop. The token is printed **only here**, so don't paste it into a chat.
Optional secret `HF_TOKEN` if the adapter repo is private.

In [ ]:
CODE_REF = "8b619ae219b47c2055f4fa65f8dc48d86431b8d0"   # exact commit of rudranaresh0201/computer-use
PORT = 8000
LONG_EDGE = 1568     # the size the eval scored
import os, subprocess, sys, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
smi = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0 and "T4" in smi.stdout, "Session options -> Accelerator -> GPU T4 x2"

In [ ]:
from pathlib import Path
REPO_DIR = Path("/kaggle/working/computer-use")
if not REPO_DIR.is_dir():
    !git clone --quiet https://github.com/rudranaresh0201/computer-use.git {REPO_DIR}
!git -C {REPO_DIR} fetch --quiet origin
!git -C {REPO_DIR} checkout --quiet {CODE_REF}
!git -C {REPO_DIR} log -1 --format="serving commit %H (%s)"
!{sys.executable} -m pip install --quiet -e "{REPO_DIR}[training]"
!{sys.executable} -m pip uninstall -y --quiet torchao
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("using the HF_TOKEN secret")
except Exception:
    print("no HF_TOKEN secret: fine if the adapter repo is public")

In [ ]:
# load once, then one smoke question on a drawn "Save" button: a point must come back, not words
import io
from PIL import Image, ImageDraw
from computeruse.eyes import Grounder

grounder = Grounder(long_edge=LONG_EDGE, device="cuda").load()
print(f"loaded in {grounder.load_seconds:.0f}s")
img = Image.new("RGB", (1568, 882), "white"); d = ImageDraw.Draw(img)
d.rectangle([1200, 40, 1320, 80], outline="black", width=2); d.text((1240, 52), "Save", fill="black")
buf = io.BytesIO(); img.save(buf, "PNG")
found = grounder.locate(buf.getvalue(), "Click Save")
print(f"smoke: {found.raw!r} -> {found.point} in {found.seconds:.1f}s  (the button spans x 1200-1320, y 40-80)")
assert found.point, "no point came back: the adapter did not load as trained"

In [ ]:
# the server, in this kernel (the model stays loaded), checked locally first
import secrets, threading
from computeruse.eyes_remote import RemoteEyes, grounder_locate, serve

TOKEN = secrets.token_urlsafe(18)
server = serve(grounder_locate(grounder), port=PORT, token=TOKEN, info={"device": "cuda", "long_edge": LONG_EDGE})
threading.Thread(target=server.serve_forever, daemon=True).start()
print("local:", RemoteEyes(f"http://127.0.0.1:{PORT}", TOKEN).health())

In [ ]:
# a public https URL for it (Cloudflare quick tunnel: no account needed)
import re
if not Path("cloudflared").exists():
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared
log = open("tunnel.log", "w")
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
                          stdout=log, stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        URL = m.group(0); break
assert URL, open("tunnel.log").read()[-2000:]
for _ in range(30):  # the name takes a few seconds to resolve
    try:
        print("through the tunnel:", RemoteEyes(URL, TOKEN).health()); break
    except Exception as e:
        time.sleep(2)
print()
print("On the laptop (PowerShell), then open a NEW terminal:")
print(f"  setx CU_EYES_URL {URL}")
print(f"  setx CU_EYES_TOKEN {TOKEN}")

In [ ]:
# keep the session busy; each request the laptop sends shows up here
while True:
    time.sleep(600)
    print(time.strftime("%H:%M"), "still serving", URL, "| tunnel alive:", tunnel.poll() is None, flush=True)